In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import to_timestamp, col, when, avg, round
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('ConfirmationRate').getOrCreate()

signups_data = [
    (3, "2020-03-21 10:16:13"),
    (7, "2020-01-04 13:57:59"),
    (2, "2020-07-29 23:09:44"),
    (6, "2020-12-09 10:39:37")
]

signups_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("time_stamp", StringType(), True)
])

signups_df = spark.createDataFrame(signups_data, signups_schema) \
    .withColumn("time_stamp", to_timestamp(col("time_stamp")))

signups_df.show()

confirmations_data = [
    (3, "2021-01-06 03:30:46", "timeout"),
    (3, "2021-07-14 14:00:00", "timeout"),
    (7, "2021-06-12 11:57:29", "confirmed"),
    (7, "2021-06-13 12:58:28", "confirmed"),
    (7, "2021-06-14 13:59:27", "confirmed"),
    (2, "2021-01-22 00:00:00", "confirmed"),
    (2, "2021-02-28 23:59:59", "timeout")
]

confirmations_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("time_stamp", StringType(), True),
    StructField("action", StringType(), True)
])

confirmations_df = spark.createDataFrame(confirmations_data, confirmations_schema) \
    .withColumn("time_stamp", to_timestamp(col("time_stamp")))

confirmations_df.show()
confirmations_df.join(signups_df , on='user_id', how='left').select(
signups_df.user_id,
    when(col('action')=='confirmed', 1).otherwise(0).alias('confirmation')
).groupBy('user_id').agg(round(avg('confirmation').alias('confirmation_rate'), 2).alias('confirmation_rate')).show()



In [0]:

from pyspark.sql.types import StructType, StructField, IntegerType

logs_data = [
    (1, 1),
    (2, 1),
    (3, 1),
    (4, 2),
    (5, 1),
    (6, 2),
    (7, 2)
]

logs_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("num", IntegerType(), True)
])

logs_df = spark.createDataFrame(logs_data, logs_schema)
a = logs_df.alias('a')
b = logs_df.alias('b')
c = logs_df.alias('c')

a.join(b, on=(a.id==b.id+1), how='inner').join(c,on=b.id+1==c.id+2, how='inner')\
.filter((a.num==b.num) & (b.num==c.num)).select(a.num).alias('num').show()


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType
from pyspark.sql import SparkSession
from pyspark.sql import functions as F, Window

logs_data = [
    (1, 1),
    (2, 1),
    (3, 1),
    (4, 2),
    (5, 1),
    (6, 2),
    (7, 2)
]

logs_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("num", IntegerType(), True)
])
logs_df = spark.createDataFrame(logs_data, logs_schema)
a = logs_df.alias('a')
b = logs_df.alias('b')
c = logs_df.alias('c')

window_spec = Window.orderBy(F.col('id'))
num_df = a.withColumn('ne_num', F.lag('num').over(window_spec))\
    .withColumn('next_nex_num', F.lag('num', 2).over(window_spec))
num_df.show()
num_df.filter((num_df.num == num_df.ne_num) & (num_df.ne_num == num_df.next_nex_num)).show()


In [0]:
"""
You receive daily transaction logs (CSV) with:
Tasks
1. Load the CSV into PySpark with an explicit schema
2. Remove duplicate transactions
3. Filter out records where amount <= 0
4. Convert transaction_timestamp to proper timestamp format
5. Calculate total debit and credit amount per day
6. Store the result in Parquet format partitioned by date


Schema:
- transaction_id (string)
- user_id (string)
- amount (double)
- transaction_type (string: credit/debit)
- transaction_timestamp (string)

"""

from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# spark = SparkSession.builder.appName('TransactionCleanup').getOrCreate()

schema = StructType([
    StructField('transaction_id', StringType(), True),
    StructField('user_id', StringType(), True),
    StructField('amount', DoubleType(), True),
    StructField('transaction_type', StringType(), True),
    StructField('transaction_timestamp', StringType(), True)

])


total_records = 1000
num_partitions = 10

df = spark.range(10, total_records,numPartitions=num_partitions)



In [0]:
from pyspark.sql import functions as F
fake_payments_df = df.select(
    
    F.expr("uuid()").alias("transaction_id"),
    F.concat(F.lit("user_"), F.floor(F.rand()*50).cast("int")).alias("user_id"),
    F.round(F.rand()*10000, 2).alias("amount"),
    F.when(F.rand() < 0.7, F.lit("debit"))
     .otherwise(F.lit("credit"))
     .alias("transaction_type"),

    (F.unix_timestamp()).cast('long')
        .alias("transaction_timestamp")

)

fake_payments_df.show(30, False)


Convert transaction_timestamp to proper timestamp format

In [0]:
payment_df = fake_payments_df.withColumn('transaction_date', F.to_date(F.col('transaction_timestamp').cast('timestamp')))
payment_df.show()

Remove duplicate transactions

In [0]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy('transaction_id').orderBy('transaction_timestamp')

ranked_df = payment_df.withColumn('rank', F.row_number().over(window_spec))

ranked_df.show()



In [0]:
ranked_df.filter(F.col('rank') == 1).show()
ranked_df.show()

 Filter out records where amount <= 0

In [0]:
amnt_filtered_df = ranked_df.filter(F.col('amount') > 1)
amnt_filtered_df.filter(F.col('user_id') == 'user_1').show()

Calculate total debit and credit amount per day

In [0]:
seggregated_amnt = amnt_filtered_df.groupBy('user_id', 'transaction_type').agg(F.round(F.sum('amount'),2).alias('amount'))
seggregated_amnt.show()




In [0]:
report_data = seggregated_amnt.groupBy('user_id').agg(
    (
        F.sum(
            F.when(
                F.col('transaction_type')=='debit', F.col('amount')
                ).otherwise(0)
        )
    ).alias('total_debit_amount'),
    (
        F.sum(
            F.when(
                F.col('transaction_type')=='credit', F.col('amount')
                ).otherwise(0)
        )
    ).alias('total_credit_amount')
)

report_data.show()

Write to Parquet Partitioned by Date

In [0]:
report_data.write \
    .mode("overwrite") \
    .partitionBy("transaction_date") \
    .parquet("path/to/output/")